Simulation Results Analysis: This notebook analyses the FSM simulation results, including state distributions, transition frequencies, transition matrices, and participant-normalized state distributions.

Imports and Paths

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load shared project paths and configuration
%run 00_config.ipynb

# =========================
# INPUT FILE
# =========================

INPUT_FILE = OUTPUT_DIR / "fsm_simulation_all_users.xlsx"

print("Simulation results found:", INPUT_FILE.exists())
print("Results folder:", RESULTS_DIR)

Load results

In [ ]:
fsm_results = pd.read_excel(INPUT_FILE)

print(f"Total records: {len(fsm_results)}")

fsm_results.head()

Create analysis datasets for state distribution and for transition analysis

In [ ]:
# Dataset for state distribution:
# We keep all rows because the onboarding initialization still represents the first assigned state for each user.
state_df = fsm_results.copy()

# Dataset for transition analysis: We remove onboarding initialization because it is not a real transition.
transition_df = fsm_results[
    fsm_results["previous_state"] != "None (Onboarding Init)"
].copy()

print(f"Records for state analysis: {len(state_df)}")
print(f"Records for transition analysis: {len(transition_df)}")

State distribution

In [ ]:
#

state_distribution = (
    state_df["new_state"]
    .value_counts()
    .reset_index()
)

state_distribution.columns = [
    "state",
    "n_days"
]

state_distribution["percentage"] = (
    state_distribution["n_days"]
    / state_distribution["n_days"].sum()
    * 100
).round(2)

state_distribution

Save state distribution

In [ ]:
state_distribution.to_excel(
    RESULTS_DIR / "01_state_distribution.xlsx",
    index=False
)

print("Saved state distribution.")

Transitions to different states summary: distinguishes actual state changes from self-transitions

In [ ]:
transition_df["is_real_transition"] = (
    transition_df["previous_state"]
    != transition_df["new_state"]
)

n_records = len(transition_df)

n_real_transitions = transition_df["is_real_transition"].sum()

n_self_transitions = n_records - n_real_transitions

transition_summary = pd.DataFrame({
    "metric": [
        "Total analysed records",
        "Real state transitions",
        "Self-transitions / state maintenance",
        "Percentage of real transitions",
        "Percentage of state maintenance"
    ],
    "value": [
        n_records,
        n_real_transitions,
        n_self_transitions,
        round(n_real_transitions / n_records * 100, 2),
        round(n_self_transitions / n_records * 100, 2)
    ]
})

transition_summary

Save transition summary

In [ ]:
transition_summary.to_excel(
    RESULTS_DIR / "02_transition_summary.xlsx",
    index=False
)

print("Saved transition summary.")

Most frequent "real" transitions: summarizes the frequency and relative percentage of actual transitions between different states.

In [ ]:
real_transitions = transition_df[
    transition_df["is_real_transition"]
].copy()

real_transitions["transition"] = (
    real_transitions["previous_state"]
    + " -> "
    + real_transitions["new_state"]
)

transition_counts = (
    real_transitions["transition"]
    .value_counts()
    .reset_index()
)

transition_counts.columns = [
    "transition",
    "count"
]

transition_counts["percentage"] = (
    transition_counts["count"]
    / transition_counts["count"].sum()
    * 100
).round(2)

transition_counts

Save transition counts

In [ ]:
transition_counts.to_excel(
    RESULTS_DIR / "03_transition_counts.xlsx",
    index=False
)

print("Saved transition counts.")

Transition matrix

In [ ]:
transition_matrix = pd.crosstab(
    transition_df["previous_state"],
    transition_df["new_state"]
)

transition_matrix

Save transition matrix in excel

In [ ]:
transition_matrix.to_excel(
    RESULTS_DIR / "04_transition_matrix.xlsx"
)

Save transition matrix in plot

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,7))

sns.heatmap(
    transition_matrix,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("FSM Transition Matrix")
plt.xlabel("Destination State")
plt.ylabel("Origin State")

plt.tight_layout()

# Save figure
plt.savefig(
    RESULTS_DIR / "04_transition_matrix_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

Participant-Normalized State Distribution

State Percentage User

In [ ]:
user_state_distribution = (
    state_df
    .groupby("user_id")["new_state"]
    .value_counts(normalize=True)
    .mul(100)
    .reset_index(name="percentage")
)

user_state_distribution.head(20)

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    mean_state_distribution["new_state"],
    mean_state_distribution["percentage"]
)

plt.ylabel("Mean Percentage Across Participants (%)")
plt.xlabel("State")
plt.title("Participant-Normalized State Distribution")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

Mean distribution per user with 0% when they dont get to a certain state

In [ ]:
pivot = (
    user_state_distribution
    .pivot(
        index="user_id",
        columns="new_state",
        values="percentage"
    )
    .fillna(0)
)

mean_state_distribution = (
    pivot
    .mean()
    .reset_index()
)

mean_state_distribution.columns = [
    "new_state",
    "percentage"
]

mean_state_distribution = (
    mean_state_distribution
    .sort_values(
        by="percentage",
        ascending=False
    )
    .reset_index(drop=True)
)

mean_state_distribution

Save participant-normalized state distribution

In [ ]:
mean_state_distribution.to_excel(
    RESULTS_DIR / "05_mean_state_distribution_per_user.xlsx",
    index=False
)

print("Saved participant-normalized state distribution.")

Participant-normalized state distribution plot

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    mean_state_distribution["new_state"],
    mean_state_distribution["percentage"]
)

plt.ylabel("Mean Percentage Across Participants (%)")
plt.xlabel("State")
plt.title("Participant-Normalized State Distribution")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()